# Home Credit — EDA Part 1 (Refined)
## Feature Inventory, Proxy Mapping, and Modeling Readiness

Notebook này giữ tinh thần của bản cũ nhưng làm gọn và thực dụng hơn cho một project portfolio cấp **Middle Data Analyst**:

- quét toàn bộ raw train tables bằng header
- nối với `feature_definitions.csv`
- gán `source_group`, `feature_family`, `proxy_concept`, `stage_candidate`
- thêm các cột hữu ích cho bước modeling:
  - `business_validated_flag`
  - `inference_availability`
  - `recommended_for_modeling`
  - `review_note`
- export ra catalog sạch để EDA Part 2 và modeling dùng tiếp

**Mục tiêu của notebook này không phải chọn model**, mà là tạo một lớp inventory đủ tốt để:
1. hiểu data landscape  
2. shortlist feature hợp lý  
3. tránh việc EDA một đằng, modeling một nẻo.

## 1. Config & path detection

In [1]:
from __future__ import annotations

import re
from pathlib import Path
import polars as pl

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(220)

# ---------------------------------------------------------
# Path detection
# ---------------------------------------------------------
def first_existing(paths: list[str]) -> Path:
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError("Cannot detect path from candidates:\n" + "\n".join(paths))

TRAIN_DIR = first_existing([
    "/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train",
    "/kaggle/input/home-credit-credit-risk-model-stability/csv_files/train",
    "/kaggle/input/csv_files/train",
])

FEATURE_DEF_PATH = first_existing([
    "/kaggle/input/notebooks/hongvittrnh/feature_definitions.csv",
    "/kaggle/input/competitions/home-credit-credit-risk-model-stability/feature_definitions.csv",
    "/kaggle/input/feature_definitions.csv",
    "/kaggle/working/feature_definitions.csv",
])

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_01")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_DIR        :", TRAIN_DIR)
print("FEATURE_DEF_PATH :", FEATURE_DEF_PATH)
print("WORK_DIR         :", WORK_DIR)

TRAIN_DIR        : /kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train
FEATURE_DEF_PATH : /kaggle/input/competitions/home-credit-credit-risk-model-stability/feature_definitions.csv
WORK_DIR         : /kaggle/working/homecredit_proxy_notebook_01


## 2. Load feature definitions

In [2]:
feature_def = (
    pl.read_csv(FEATURE_DEF_PATH)
    .rename({"Variable": "variable", "Description": "description"})
    .with_columns([
        pl.col("variable").cast(pl.Utf8).str.strip_chars(),
        pl.col("description").cast(pl.Utf8).fill_null("").str.strip_chars(),
    ])
    .unique(subset=["variable"])
    .sort("variable")
)

print("feature_def shape:", feature_def.shape)
display(feature_def.head(10))

feature_def shape: (465, 2)


variable,description
str,str
"""actualdpd_943P""","""Days Past Due (DPD) of previous contract (actual)."""
"""actualdpdtolerance_344P""","""DPD of client with tolerance."""
"""addres_district_368M""","""District of the person's address."""
"""addres_role_871L""","""Role of person's address."""
"""addres_zip_823M""","""Zip code of the address."""
"""amount_1115A""","""Credit amount of the active contract provided by the credit bureau."""
"""amount_416A""","""Deposit amount."""
"""amount_4527230A""","""Tax deductions amount tracked by the government registry."""
"""amount_4917619A""","""Tax deductions amount tracked by the government registry."""


## 3. Scan raw train tables by header only

In [3]:
def list_train_files(base_dir: Path) -> pl.DataFrame:
    rows = []
    for path in sorted(base_dir.glob("*.csv")):
        rows.append({
            "file_name": path.name,
            "file_path": str(path),
            "table_tag": path.stem,
        })
    if not rows:
        raise FileNotFoundError(f"No CSV files found under: {base_dir}")
    return pl.DataFrame(rows)

def read_csv_header(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    return [c.strip() for c in first_line.split(",")]

def scan_headers(file_catalog: pl.DataFrame) -> pl.DataFrame:
    rows = []
    for r in file_catalog.iter_rows(named=True):
        cols = read_csv_header(r["file_path"])
        for idx, col in enumerate(cols):
            rows.append({
                "file_name": r["file_name"],
                "table_tag": r["table_tag"],
                "column_name": col,
                "col_order": idx,
            })
    return pl.DataFrame(rows)

file_catalog = list_train_files(TRAIN_DIR)
header_inventory = scan_headers(file_catalog)

print("file_catalog shape   :", file_catalog.shape)
print("header_inventory shape:", header_inventory.shape)
display(file_catalog.head(10))
display(header_inventory.head(10))

file_catalog shape   : (32, 3)
header_inventory shape: (1139, 4)


file_name,file_path,table_tag
str,str,str
"""train_applprev_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv""","""train_applprev_1_0"""
"""train_applprev_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_1.csv""","""train_applprev_1_1"""
"""train_applprev_2.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_2.csv""","""train_applprev_2"""
"""train_base.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_base.csv""","""train_base"""
"""train_credit_bureau_a_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_0.csv""","""train_credit_bureau_a_1_0"""
"""train_credit_bureau_a_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_1.csv""","""train_credit_bureau_a_1_1"""
"""train_credit_bureau_a_1_2.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_2.csv""","""train_credit_bureau_a_1_2"""
"""train_credit_bureau_a_1_3.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_3.csv""","""train_credit_bureau_a_1_3"""
"""train_credit_bureau_a_2_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_2_0.csv""","""train_credit_bureau_a_2_0"""


file_name,table_tag,column_name,col_order
str,str,str,i64
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""case_id""",0
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""actualdpd_943P""",1
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""annuity_853A""",2
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""approvaldate_319D""",3
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""byoccupationinc_3656910L""",4
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""cancelreason_3545846M""",5
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""childnum_21L""",6
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""creationdate_885D""",7
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""credacc_actualbalance_314A""",8


## 4. Build feature catalog from definitions + header inventory

In [4]:
header_agg = (
    header_inventory
    .group_by("column_name")
    .agg([
        pl.len().alias("n_tables"),
        pl.col("table_tag").sort().alias("source_tables"),
        pl.col("file_name").sort().alias("source_files"),
    ])
    .rename({"column_name": "variable"})
)

feature_catalog = (
    feature_def
    .join(header_agg, on="variable", how="outer_coalesce")
    .with_columns([
        pl.col("description").fill_null(""),
        pl.col("n_tables").fill_null(0),
        pl.when(pl.col("n_tables").fill_null(0) > 0)
          .then(pl.lit(True))
          .otherwise(pl.lit(False))
          .alias("exists_in_raw"),
    ])
    .sort(["exists_in_raw", "variable"], descending=[True, False])
)

print("feature_catalog shape:", feature_catalog.shape)
display(feature_catalog.head(20))

feature_catalog shape: (472, 6)


/tmp/ipykernel_17/2985733270.py:14: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  .join(header_agg, on="variable", how="outer_coalesce")


variable,description,n_tables,source_tables,source_files,exists_in_raw
str,str,u32,list[str],list[str],bool
"""MONTH""","""""",1,"[""train_base""]","[""train_base.csv""]",true
"""WEEK_NUM""","""""",1,"[""train_base""]","[""train_base.csv""]",true
"""actualdpd_943P""","""Days Past Due (DPD) of previous contract (actual).""",2,"[""train_applprev_1_0"", ""train_applprev_1_1""]","[""train_applprev_1_0.csv"", ""train_applprev_1_1.csv""]",true
"""actualdpdtolerance_344P""","""DPD of client with tolerance.""",2,"[""train_static_0_0"", ""train_static_0_1""]","[""train_static_0_0.csv"", ""train_static_0_1.csv""]",true
"""addres_district_368M""","""District of the person's address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true
"""addres_role_871L""","""Role of person's address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true
"""addres_zip_823M""","""Zip code of the address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true
"""amount_1115A""","""Credit amount of the active contract provided by the credit bureau.""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]",true
"""amount_416A""","""Deposit amount.""",1,"[""train_deposit_1""]","[""train_deposit_1.csv""]",true


## 5. Parse suffix type and create source/family/business heuristics

In [5]:
SUFFIX_PATTERN = r"_([0-9]+)([A-Z])$"

def suffix_letter_expr(col: str = "variable") -> pl.Expr:
    return pl.col(col).str.extract(SUFFIX_PATTERN, group_index=2)

def suffix_number_expr(col: str = "variable") -> pl.Expr:
    return pl.col(col).str.extract(SUFFIX_PATTERN, group_index=1).cast(pl.Int64, strict=False)

def source_tables_str_expr() -> pl.Expr:
    return (
        pl.when(pl.col("source_tables").is_null())
        .then(pl.lit(""))
        .otherwise(pl.col("source_tables").list.join(" | "))
    )

def infer_source_group_expr() -> pl.Expr:
    src = source_tables_str_expr().str.to_lowercase()
    return (
        pl.when(src.str.contains("bureau", literal=True)).then(pl.lit("bureau"))
        .when(src.str.contains("applprev", literal=True)).then(pl.lit("previous_application"))
        .when(src.str.contains("deposit", literal=True)).then(pl.lit("deposit"))
        .when(src.str.contains("tax", literal=True)).then(pl.lit("tax_registry"))
        .when(src.str.contains("person", literal=True)).then(pl.lit("person"))
        .when(src.str.contains("static", literal=True)).then(pl.lit("application_static"))
        .when(src.str.contains("credit", literal=True)).then(pl.lit("credit_internal"))
        .when(src.str.contains("install", literal=True)).then(pl.lit("installments"))
        .otherwise(pl.lit("other"))
    )

def infer_family_expr() -> pl.Expr:
    v = pl.col("variable").fill_null("").str.to_lowercase()
    d = pl.col("description").fill_null("").str.to_lowercase()
    txt = pl.concat_str([v, pl.lit(" "), d], separator="")

    return (
        pl.when(txt.str.contains("dpd|delinq|overdue|past due|late", literal=False)).then(pl.lit("delinquency"))
        .when(txt.str.contains("annuity|install|instal|payment", literal=False)).then(pl.lit("repayment"))
        .when(txt.str.contains("deposit|income|salary|amount", literal=False)).then(pl.lit("capacity"))
        .when(txt.str.contains("contract|credit|debt|bureau|exposure", literal=False)).then(pl.lit("credit_burden"))
        .when(txt.str.contains("application|address|phone|mobile|contact", literal=False)).then(pl.lit("identity_stability"))
        .when(txt.str.contains("date|days|month|week|year", literal=False)).then(pl.lit("date_or_recency"))
        .otherwise(pl.lit("other"))
    )

def infer_proxy_concept_expr() -> pl.Expr:
    fam = pl.col("feature_family")
    src = pl.col("source_group")
    v = pl.col("variable").fill_null("").str.to_lowercase()

    return (
        pl.when((fam == "capacity") & src.is_in(["deposit", "tax_registry", "application_static"]))
        .then(pl.lit("capacity_to_repay"))
        .when((fam == "credit_burden") & src.is_in(["bureau", "credit_internal", "previous_application"]))
        .then(pl.lit("credit_burden"))
        .when(fam == "delinquency")
        .then(pl.lit("delinquency_risk"))
        .when((fam == "repayment") & v.str.contains("paidbefdue|befdue|timely", literal=False))
        .then(pl.lit("willingness_to_pay"))
        .when(fam == "identity_stability")
        .then(pl.lit("identity_instability"))
        .when(fam == "date_or_recency")
        .then(pl.lit("recency_or_tenure"))
        .otherwise(pl.lit("other_proxy"))
    )

def infer_stage_candidate_expr() -> pl.Expr:
    pc = pl.col("proxy_concept")
    fam = pl.col("feature_family")
    return (
        pl.when(pc.is_in(["capacity_to_repay", "credit_burden", "identity_instability"]))
        .then(pl.lit("A_or_B"))
        .when(pc == "willingness_to_pay")
        .then(pl.lit("B_or_C"))
        .when((pc == "delinquency_risk") | (fam == "delinquency"))
        .then(pl.lit("B_or_C"))
        .when(pc == "recency_or_tenure")
        .then(pl.lit("review_first"))
        .otherwise(pl.lit("review_first"))
    )

def infer_priority_expr() -> pl.Expr:
    pc = pl.col("proxy_concept")
    fam = pl.col("feature_family")
    exists = pl.col("exists_in_raw")
    return (
        pl.when(~exists).then(pl.lit("drop_no_raw"))
        .when(pc.is_in(["capacity_to_repay", "credit_burden", "willingness_to_pay", "delinquency_risk"]))
        .then(pl.lit("high"))
        .when(fam.is_in(["identity_stability", "repayment"]))
        .then(pl.lit("medium"))
        .otherwise(pl.lit("low"))
    )

def infer_leakage_risk_expr() -> pl.Expr:
    txt = pl.concat_str([
        pl.col("variable").fill_null("").str.to_lowercase(),
        pl.lit(" "),
        pl.col("description").fill_null("").str.to_lowercase(),
        pl.lit(" "),
        source_tables_str_expr().str.to_lowercase()
    ], separator="")

    return (
        pl.when(txt.str.contains("closure|closed|post|recovery|write off|last.*dpd|final", literal=False))
        .then(pl.lit("high_review"))
        .when(txt.str.contains("date|days", literal=False))
        .then(pl.lit("medium_review"))
        .otherwise(pl.lit("low"))
    )

feature_catalog = (
    feature_catalog
    .with_columns([
        suffix_letter_expr().alias("suffix_type"),
        suffix_number_expr().alias("suffix_id"),
        infer_source_group_expr().alias("source_group"),
    ])
    .with_columns([
        infer_family_expr().alias("feature_family"),
    ])
    .with_columns([
        infer_proxy_concept_expr().alias("proxy_concept"),
    ])
    .with_columns([
        infer_stage_candidate_expr().alias("stage_candidate"),
        infer_priority_expr().alias("priority"),
        infer_leakage_risk_expr().alias("leakage_risk"),
    ])
)

display(
    feature_catalog.select([
        "variable",
        "source_group",
        "feature_family",
        "proxy_concept",
        "stage_candidate",
        "priority",
        "leakage_risk",
        "exists_in_raw",
    ]).head(20)
)

variable,source_group,feature_family,proxy_concept,stage_candidate,priority,leakage_risk,exists_in_raw
str,str,str,str,str,str,str,bool
"""MONTH""","""other""","""date_or_recency""","""recency_or_tenure""","""review_first""","""low""","""low""",true
"""WEEK_NUM""","""other""","""date_or_recency""","""recency_or_tenure""","""review_first""","""low""","""low""",true
"""actualdpd_943P""","""previous_application""","""delinquency""","""delinquency_risk""","""B_or_C""","""high""","""medium_review""",true
"""actualdpdtolerance_344P""","""application_static""","""delinquency""","""delinquency_risk""","""B_or_C""","""high""","""low""",true
"""addres_district_368M""","""person""","""identity_stability""","""identity_instability""","""A_or_B""","""medium""","""low""",true
"""addres_role_871L""","""person""","""identity_stability""","""identity_instability""","""A_or_B""","""medium""","""low""",true
"""addres_zip_823M""","""person""","""identity_stability""","""identity_instability""","""A_or_B""","""medium""","""low""",true
"""amount_1115A""","""bureau""","""capacity""","""other_proxy""","""review_first""","""low""","""low""",true
"""amount_416A""","""deposit""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true


## 6. Manual seed map for strong business proxies

In [6]:
seed_proxy_map = {
    # A / application style
    "amtdepositincoming_4809444A": ("capacity_to_repay", "A_or_B", "high", "good A proxy from deposit flow"),
    "amtdepositbalance_4809441A": ("capacity_to_repay", "A_or_B", "high", "balance proxy"),
    "amount_4527230A": ("capacity_to_repay", "A_or_B", "high", "tax / amount proxy"),
    "amount_4917619A": ("capacity_to_repay", "A_or_B", "high", "tax / amount proxy"),
    "contractssum_5085716L": ("credit_burden", "A_or_B", "high", "bureau burden proxy"),
    "credquantity_1099L": ("credit_burden", "A_or_B", "high", "number of credits"),
    "amount_1115A": ("credit_burden", "A_or_B", "high", "active debt amount"),
    "applicationcnt_361L": ("identity_instability", "A_or_B", "high", "repeat application signal"),

    # B / early behavior
    "avgmaxdpdlast9m_3716943P": ("delinquency_risk", "B_or_C", "high", "recent delinquency"),
    "actualdpd_943P": ("delinquency_risk", "B_or_C", "high", "current DPD"),
    "amtinstpaidbefduel24m_4187115A": ("willingness_to_pay", "B_or_C", "high", "paid before due"),
    "annuity_853A": ("credit_burden", "A_or_B", "high", "short-term repayment burden"),

    # C / collection
    "avgdbddpdlast24m_3658932P": ("delinquency_risk", "C_only", "high", "longer delinquency pattern"),
    "avgdpdtolclosure24_3658938P": ("delinquency_risk", "C_only", "high", "closure delinquency proxy"),
    "datelastinstal40dpd_247D": ("recency_or_tenure", "C_only", "medium", "last 40+ DPD recency"),
}

seed_df = pl.DataFrame(
    [{
        "variable": k,
        "seed_proxy_concept": v[0],
        "seed_stage_candidate": v[1],
        "seed_priority": v[2],
        "seed_note": v[3],
    } for k, v in seed_proxy_map.items()]
)

display(seed_df)

variable,seed_proxy_concept,seed_stage_candidate,seed_priority,seed_note
str,str,str,str,str
"""amtdepositincoming_4809444A""","""capacity_to_repay""","""A_or_B""","""high""","""good A proxy from deposit flow"""
"""amtdepositbalance_4809441A""","""capacity_to_repay""","""A_or_B""","""high""","""balance proxy"""
"""amount_4527230A""","""capacity_to_repay""","""A_or_B""","""high""","""tax / amount proxy"""
"""amount_4917619A""","""capacity_to_repay""","""A_or_B""","""high""","""tax / amount proxy"""
"""contractssum_5085716L""","""credit_burden""","""A_or_B""","""high""","""bureau burden proxy"""
"""credquantity_1099L""","""credit_burden""","""A_or_B""","""high""","""number of credits"""
"""amount_1115A""","""credit_burden""","""A_or_B""","""high""","""active debt amount"""
"""applicationcnt_361L""","""identity_instability""","""A_or_B""","""high""","""repeat application signal"""
"""avgmaxdpdlast9m_3716943P""","""delinquency_risk""","""B_or_C""","""high""","""recent delinquency"""


## 7. Add practical modeling flags

In [7]:
feature_catalog = (
    feature_catalog
    .join(seed_df, on="variable", how="left")
    .with_columns([
        pl.coalesce([pl.col("seed_proxy_concept"), pl.col("proxy_concept")]).alias("proxy_concept_final"),
        pl.coalesce([pl.col("seed_stage_candidate"), pl.col("stage_candidate")]).alias("stage_candidate_final"),
        pl.coalesce([pl.col("seed_priority"), pl.col("priority")]).alias("priority_final"),
        pl.col("seed_note").fill_null("").alias("review_note"),
    ])
    .with_columns([
        pl.when(pl.col("priority_final").is_in(["high", "medium"]) & (pl.col("leakage_risk") != "high_review"))
          .then(pl.lit(True))
          .otherwise(pl.lit(False))
          .alias("business_validated_flag"),

        pl.when(pl.col("stage_candidate_final") == "C_only").then(pl.lit("needs_internal_history"))
          .when(pl.col("stage_candidate_final").is_in(["B_or_C", "A_or_B"])).then(pl.lit("batch_scoring_ok"))
          .otherwise(pl.lit("review"))
          .alias("inference_availability"),

        pl.when(
            pl.col("exists_in_raw")
            & pl.col("priority_final").is_in(["high", "medium"])
            & (pl.col("leakage_risk") != "high_review")
        ).then(pl.lit(True)).otherwise(pl.lit(False)).alias("recommended_for_modeling"),
    ])
)

display(
    feature_catalog
    .select([
        "variable", "proxy_concept_final", "stage_candidate_final", "priority_final",
        "business_validated_flag", "inference_availability", "recommended_for_modeling", "review_note"
    ])
    .head(25)
)

variable,proxy_concept_final,stage_candidate_final,priority_final,business_validated_flag,inference_availability,recommended_for_modeling,review_note
str,str,str,str,bool,str,bool,str
"""MONTH""","""recency_or_tenure""","""review_first""","""low""",false,"""review""",false,""""""
"""WEEK_NUM""","""recency_or_tenure""","""review_first""","""low""",false,"""review""",false,""""""
"""actualdpd_943P""","""delinquency_risk""","""B_or_C""","""high""",true,"""batch_scoring_ok""",true,"""current DPD"""
"""actualdpdtolerance_344P""","""delinquency_risk""","""B_or_C""","""high""",true,"""batch_scoring_ok""",true,""""""
"""addres_district_368M""","""identity_instability""","""A_or_B""","""medium""",true,"""batch_scoring_ok""",true,""""""
"""addres_role_871L""","""identity_instability""","""A_or_B""","""medium""",true,"""batch_scoring_ok""",true,""""""
"""addres_zip_823M""","""identity_instability""","""A_or_B""","""medium""",true,"""batch_scoring_ok""",true,""""""
"""amount_1115A""","""credit_burden""","""A_or_B""","""high""",true,"""batch_scoring_ok""",true,"""active debt amount"""
"""amount_416A""","""capacity_to_repay""","""A_or_B""","""high""",true,"""batch_scoring_ok""",true,""""""


## 8. Build proxy candidate catalog for review

In [8]:
proxy_candidate_catalog = (
    feature_catalog
    .filter(pl.col("recommended_for_modeling") == True)
    .select([
        "variable",
        "description",
        "source_group",
        "feature_family",
        "proxy_concept_final",
        "stage_candidate_final",
        "priority_final",
        "leakage_risk",
        "business_validated_flag",
        "inference_availability",
        "review_note",
        "n_tables",
        "source_tables",
        "source_files",
    ])
    .sort(["priority_final", "stage_candidate_final", "variable"])
)

print("proxy_candidate_catalog shape:", proxy_candidate_catalog.shape)
display(proxy_candidate_catalog.head(30))

proxy_candidate_catalog shape: (276, 14)


variable,description,source_group,feature_family,proxy_concept_final,stage_candidate_final,priority_final,leakage_risk,business_validated_flag,inference_availability,review_note,n_tables,source_tables,source_files
str,str,str,str,str,str,str,str,bool,str,str,u32,list[str],list[str]
"""amount_1115A""","""Credit amount of the active contract provided by the credit bureau.""","""bureau""","""capacity""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""active debt amount""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]"
"""amount_416A""","""Deposit amount.""","""deposit""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",1,"[""train_deposit_1""]","[""train_deposit_1.csv""]"
"""amount_4527230A""","""Tax deductions amount tracked by the government registry.""","""tax_registry""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""tax / amount proxy""",1,"[""train_tax_registry_a_1""]","[""train_tax_registry_a_1.csv""]"
"""amount_4917619A""","""Tax deductions amount tracked by the government registry.""","""tax_registry""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""tax / amount proxy""",1,"[""train_tax_registry_b_1""]","[""train_tax_registry_b_1.csv""]"
"""amtdepositbalance_4809441A""","""Deposit balance of client.""","""other""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""balance proxy""",1,"[""train_other_1""]","[""train_other_1.csv""]"
"""amtdepositincoming_4809444A""","""Amount of incoming deposits to client's account.""","""other""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""good A proxy from deposit flow""",1,"[""train_other_1""]","[""train_other_1.csv""]"
"""annualeffectiverate_63L""","""Interest rate for the active contracts.""","""bureau""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",4,"[""train_credit_bureau_a_1_0"", ""train_credit_bureau_a_1_1"", … ""train_credit_bureau_a_1_3""]","[""train_credit_bureau_a_1_0.csv"", ""train_credit_bureau_a_1_1.csv"", … ""train_credit_bureau_a_1_3.csv""]"
"""annuity_853A""","""Monthly annuity for previous applications.""","""previous_application""","""repayment""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""short-term repayment burden""",2,"[""train_applprev_1_0"", ""train_applprev_1_1""]","[""train_applprev_1_0.csv"", ""train_applprev_1_1.csv""]"
"""applicationcnt_361L""","""Number of applications associated with the same email address as the client.""","""application_static""","""identity_stability""","""identity_instability""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""repeat application signal""",2,"[""train_static_0_0"", ""train_static_0_1""]","[""train_static_0_0.csv"", ""train_static_0_1.csv""]"


## 9. Review summaries

In [9]:
summary_stage = (
    proxy_candidate_catalog
    .group_by("stage_candidate_final")
    .agg([
        pl.len().alias("n_features"),
        pl.col("variable").n_unique().alias("n_unique_features"),
    ])
    .sort("n_features", descending=True)
)

summary_source = (
    proxy_candidate_catalog
    .group_by("source_group")
    .agg(pl.len().alias("n_features"))
    .sort("n_features", descending=True)
)

summary_family = (
    proxy_candidate_catalog
    .group_by("feature_family")
    .agg(pl.len().alias("n_features"))
    .sort("n_features", descending=True)
)

print("=== By stage ===")
display(summary_stage)
print("=== By source ===")
display(summary_source)
print("=== By family ===")
display(summary_family)

=== By stage ===


stage_candidate_final,n_features,n_unique_features
str,u32,u32
"""A_or_B""",144,144
"""review_first""",67,67
"""B_or_C""",64,64
"""C_only""",1,1


=== By source ===


source_group,n_features
str,u32
"""application_static""",133
"""bureau""",82
"""previous_application""",36
"""person""",17
"""deposit""",3
"""tax_registry""",3
"""other""",2


=== By family ===


feature_family,n_features
str,u32
"""identity_stability""",76
"""repayment""",69
"""delinquency""",64
"""credit_burden""",44
"""capacity""",23


## 10. Recommended review slices

In [10]:
print("=== A / B practical candidates ===")
display(
    proxy_candidate_catalog
    .filter(pl.col("stage_candidate_final").is_in(["A_or_B", "B_or_C"]))
    .select(["variable", "proxy_concept_final", "stage_candidate_final", "priority_final", "source_group", "review_note"])
    .head(30)
)

print("=== C-heavy candidates ===")
display(
    proxy_candidate_catalog
    .filter(pl.col("stage_candidate_final") == "C_only")
    .select(["variable", "proxy_concept_final", "priority_final", "source_group", "review_note"])
    .head(30)
)

=== A / B practical candidates ===


variable,proxy_concept_final,stage_candidate_final,priority_final,source_group,review_note
str,str,str,str,str,str
"""amount_1115A""","""credit_burden""","""A_or_B""","""high""","""bureau""","""active debt amount"""
"""amount_416A""","""capacity_to_repay""","""A_or_B""","""high""","""deposit""",""""""
"""amount_4527230A""","""capacity_to_repay""","""A_or_B""","""high""","""tax_registry""","""tax / amount proxy"""
"""amount_4917619A""","""capacity_to_repay""","""A_or_B""","""high""","""tax_registry""","""tax / amount proxy"""
"""amtdepositbalance_4809441A""","""capacity_to_repay""","""A_or_B""","""high""","""other""","""balance proxy"""
"""amtdepositincoming_4809444A""","""capacity_to_repay""","""A_or_B""","""high""","""other""","""good A proxy from deposit flow"""
"""annualeffectiverate_63L""","""credit_burden""","""A_or_B""","""high""","""bureau""",""""""
"""annuity_853A""","""credit_burden""","""A_or_B""","""high""","""previous_application""","""short-term repayment burden"""
"""applicationcnt_361L""","""identity_instability""","""A_or_B""","""high""","""application_static""","""repeat application signal"""


=== C-heavy candidates ===


variable,proxy_concept_final,priority_final,source_group,review_note
str,str,str,str,str
"""avgdbddpdlast24m_3658932P""","""delinquency_risk""","""high""","""application_static""","""longer delinquency pattern"""


## 11. Export outputs for EDA Part 2

In [11]:
feature_catalog_path = WORK_DIR / "feature_catalog.parquet"
proxy_catalog_path   = WORK_DIR / "proxy_candidate_catalog.parquet"
header_inv_path      = WORK_DIR / "header_inventory.parquet"
file_catalog_path    = WORK_DIR / "file_catalog.parquet"

feature_catalog.write_parquet(feature_catalog_path)
proxy_candidate_catalog.write_parquet(proxy_catalog_path)
header_inventory.write_parquet(header_inv_path)
file_catalog.write_parquet(file_catalog_path)

print("[SAVED]", feature_catalog_path)
print("[SAVED]", proxy_catalog_path)
print("[SAVED]", header_inv_path)
print("[SAVED]", file_catalog_path)

[SAVED] /kaggle/working/homecredit_proxy_notebook_01/feature_catalog.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_01/proxy_candidate_catalog.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_01/header_inventory.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_01/file_catalog.parquet


## 12. Output of this notebook

Sau notebook này bạn đã có:
- `feature_catalog.parquet`
- `proxy_candidate_catalog.parquet`
- `header_inventory.parquet`
- `file_catalog.parquet`

Notebook EDA Part 2 sẽ dùng các file này để:
- validate stage rule A / B / C
- chọn anchor date phù hợp hơn
- profile feature theo stage
- tạo shortlist thực dụng cho modeling notebook